In [1]:
import torch
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kurtosis
from marketsim.simulator.simulator import Simulator
from marketsim.simulator.simulator_v2 import Simulator as Simulator2

In [2]:
def load_stockmarl_close_series(csv_path, ticker="AAPL"):
    raw = pd.read_csv(csv_path)

    # row 0 contains ticker labels
    ticker_row = raw.iloc[0]

    # actual data starts at row 2
    data = raw.iloc[2:].copy().reset_index(drop=True)

    # find the Close column for the requested ticker
    close_cols = [col for col in raw.columns if str(col).startswith("Close")]
    selected_col = None

    for col in close_cols:
        if ticker_row[col] == ticker:
            selected_col = col
            break

    if selected_col is None:
        raise ValueError(f"Could not find Close column for ticker {ticker}")

    prices = pd.to_numeric(data[selected_col], errors="coerce").dropna().to_numpy(dtype=float)
    return prices

csv_path = r"R:\sescott1\Masters\thesis\potential_codes\StockMARL\Stock-MARL-main\resources\datasets\train_dataV1.csv"
ticker = "AAPL"
sim_time = 500

historical_prices = load_stockmarl_close_series(csv_path, ticker=ticker)

In [3]:
def run_simulation(sim_type, seed, sim_time, historical_prices):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if sim_type == "baseline":
        sim = Simulator(
            num_background_agents=50,
            sim_time=sim_time,
            num_assets=1,
            lam=0.1,
            mean=100.0,
            r=0.6,
            shock_var=10.0,
            q_max=10,
            pv_var=5e6,
            zi_shade=[10, 30],
            seed=seed
        )

    elif sim_type == "rescaled":
        rescaled_mean = float(np.mean(historical_prices[:sim_time + 1]))
        sim = Simulator(
            num_background_agents=50,
            sim_time=sim_time,
            num_assets=1,
            lam=0.1,
            mean=rescaled_mean,
            r=0.6,
            shock_var=1.0,
            q_max=10,
            pv_var=1.0,
            zi_shade=[0.05, 0.5],
            seed=seed
        )

    elif sim_type == "historical":
        sim = Simulator2(
            num_background_agents=50,
            sim_time=sim_time,
            num_assets=1,
            lam=0.1,
            q_max=10,
            pv_var=1.0,
            zi_shade=[0.05, 0.5],
            fundamental_type="historical",
            historical_prices=historical_prices,
            seed=seed
        )

    else:
        raise ValueError(f"Unknown sim_type: {sim_type}")

    sim.run()

    market = sim.markets[0]
    midprices = np.array(market.get_midprices(), dtype=float)
    fundamental = np.array(
        [market.fundamental.get_value_at(t) for t in range(len(midprices))],
        dtype=float
    )

    return market, midprices, fundamental

In [4]:
def safe_log_returns(price_series):
    price_series = np.asarray(price_series, dtype=float)
    price_series = price_series[np.isfinite(price_series)]
    if len(price_series) < 2:
        return np.array([])
    if np.any(price_series <= 0):
        return np.array([])
    return np.diff(np.log(price_series))


def lag1_autocorr(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return np.nan
    return np.corrcoef(x[:-1], x[1:])[0, 1]


def compare_to_real(sim_prices, real_prices):
    sim_prices = np.asarray(sim_prices, dtype=float)
    real_prices = np.asarray(real_prices, dtype=float)

    n = min(len(sim_prices), len(real_prices))
    sim_prices = sim_prices[:n]
    real_prices = real_prices[:n]

    errors = sim_prices - real_prices
    return {
        "mae_vs_real": np.mean(np.abs(errors)),
        "rmse_vs_real": np.sqrt(np.mean(errors**2)),
        "corr_vs_real": np.corrcoef(sim_prices, real_prices)[0, 1],
        "error_std_vs_real": np.std(errors),
    }

In [5]:
def compute_run_metrics(sim_type, seed, market, midprices, fundamental, real_prices):
    # align simulated prices to real prices
    n = min(len(midprices), len(real_prices))
    midprices_aligned = midprices[:n]
    fundamental_aligned = fundamental[:n]
    real_prices_aligned = real_prices[:n]

    # level tracking metrics
    vs_real = compare_to_real(midprices_aligned, real_prices_aligned)

    # returns metrics
    sim_ret = safe_log_returns(midprices_aligned)

    if len(sim_ret) == 0:
        mean_return = np.nan
        std_return = np.nan
        kurt = np.nan
        acf_sq = np.nan
        acf_abs = np.nan
        n_returns = 0
    else:
        mean_return = np.mean(sim_ret)
        std_return = np.std(sim_ret)
        kurt = kurtosis(sim_ret, fisher=False)
        acf_sq = lag1_autocorr(sim_ret**2)
        acf_abs = lag1_autocorr(np.abs(sim_ret))
        n_returns = len(sim_ret)

    # tracking to its own fundamental too
    own_tracking = compare_to_real(midprices_aligned, fundamental_aligned)

    row = {
        "sim_type": sim_type,
        "seed": seed,
        "matched_orders": len(market.matched_orders),
        "n_returns": n_returns,
        "mean_return": mean_return,
        "std_return": std_return,
        "kurtosis": kurt,
        "lag1_acf_squared_returns": acf_sq,
        "lag1_acf_abs_returns": acf_abs,
        "mae_vs_real": vs_real["mae_vs_real"],
        "rmse_vs_real": vs_real["rmse_vs_real"],
        "corr_vs_real": vs_real["corr_vs_real"],
        "error_std_vs_real": vs_real["error_std_vs_real"],
        "mae_vs_own_fundamental": own_tracking["mae_vs_real"],
        "rmse_vs_own_fundamental": own_tracking["rmse_vs_real"],
        "corr_vs_own_fundamental": own_tracking["corr_vs_real"],
        "error_std_vs_own_fundamental": own_tracking["error_std_vs_real"],
    }

    return row

In [6]:
real_prices = np.asarray(historical_prices[:sim_time + 1], dtype=float)
real_ret = safe_log_returns(real_prices)

real_metrics = {
    "n_returns": len(real_ret),
    "mean_return": np.mean(real_ret),
    "std_return": np.std(real_ret),
    "kurtosis": kurtosis(real_ret, fisher=False),
    "lag1_acf_squared_returns": lag1_autocorr(real_ret**2),
    "lag1_acf_abs_returns": lag1_autocorr(np.abs(real_ret)),
}

real_metrics

{'n_returns': 500,
 'mean_return': -7.10363074049134e-05,
 'std_return': 0.016054193345726803,
 'kurtosis': 5.2524051732383,
 'lag1_acf_squared_returns': 0.10124966124722887,
 'lag1_acf_abs_returns': 0.12967233649975016}

In [7]:
results = []

for sim_type in ["baseline", "rescaled", "historical"]:
    for seed in range(20):
        market, midprices, fundamental = run_simulation(
            sim_type=sim_type,
            seed=seed,
            sim_time=sim_time,
            historical_prices=historical_prices
        )

        row = compute_run_metrics(
            sim_type=sim_type,
            seed=seed,
            market=market,
            midprices=midprices,
            fundamental=fundamental,
            real_prices=real_prices
        )

        results.append(row)

results_df = pd.DataFrame(results)

In [8]:
mean_results_df = results_df.groupby("sim_type").mean(numeric_only=True)
print("Mean Results:")
print(mean_results_df)

std_results_df = results_df.groupby("sim_type").std(numeric_only=True)
print("\nStandard Deviation of Results:")
print(std_results_df)

summary_mean_std = results_df.groupby("sim_type").agg(["mean", "std"])
#print(summary_mean_std)

actual_row = pd.DataFrame(real_metrics, index=["actual_historical"])
#actual_row

Mean Results:
            seed  matched_orders  n_returns  mean_return  std_return  \
sim_type                                                               
baseline     9.5           140.9      24.90    -0.003188    0.100168   
historical   9.5          1071.9     498.50    -0.000069    0.008925   
rescaled     9.5            66.6     498.55     0.000002    0.000972   

             kurtosis  lag1_acf_squared_returns  lag1_acf_abs_returns  \
sim_type                                                                
baseline    26.039927                  0.392110              0.442129   
historical  28.443258                  0.041064              0.118695   
rescaled    21.734378                  0.114693              0.097956   

            mae_vs_real  rmse_vs_real  corr_vs_real  error_std_vs_real  \
sim_type                                                                 
baseline      96.738664    108.681935     -0.012825          51.671884   
historical     0.438778      0.630056

In [9]:
cols = [
    "matched_orders", "n_returns", "mean_return", "std_return", "kurtosis",
    "lag1_acf_squared_returns", "lag1_acf_abs_returns",
    "mae_vs_real", "rmse_vs_real", "corr_vs_real", "error_std_vs_real",
    "mae_vs_own_fundamental", "rmse_vs_own_fundamental",
    "corr_vs_own_fundamental", "error_std_vs_own_fundamental"
]

mean_results_clean = results_df.groupby("sim_type")[cols].mean()
std_results_clean = results_df.groupby("sim_type")[cols].std()
print("Mean Results (Clean):")
print(mean_results_clean)
print("\nStandard Deviation of Results (Clean):")
print(std_results_clean)

Mean Results (Clean):
            matched_orders  n_returns  mean_return  std_return   kurtosis  \
sim_type                                                                    
baseline             140.9      24.90    -0.003188    0.100168  26.039927   
historical          1071.9     498.50    -0.000069    0.008925  28.443258   
rescaled              66.6     498.55     0.000002    0.000972  21.734378   

            lag1_acf_squared_returns  lag1_acf_abs_returns  mae_vs_real  \
sim_type                                                                  
baseline                    0.392110              0.442129    96.738664   
historical                  0.041064              0.118695     0.438778   
rescaled                    0.114693              0.097956     2.234462   

            rmse_vs_real  corr_vs_real  error_std_vs_real  \
sim_type                                                    
baseline      108.681935     -0.012825          51.671884   
historical      0.630056      0.9

In [10]:
cols_to_show = [
    "mae_vs_real",
    "rmse_vs_real",
    "corr_vs_real",
    "std_return",
    "kurtosis",
    "lag1_acf_squared_returns",
    "lag1_acf_abs_returns",
    "matched_orders"
]

mean_results_df[cols_to_show]

,mae_vs_real,rmse_vs_real,corr_vs_real,std_return,kurtosis,lag1_acf_squared_returns,lag1_acf_abs_returns,matched_orders
sim_type,,,,,,,,
baseline,96.738664,108.681935,-0.012825,0.100168,26.039927,0.392110,0.442129,140.9
historical,0.438778,0.630056,0.972769,0.008925,28.443258,0.041064,0.118695,1071.9
rescaled,2.234462,2.727927,-0.057219,0.000972,21.734378,0.114693,0.097956,66.6


In [11]:
from marketsim.wrappers.MM_wrapper_v2 import MMEnv
import numpy as np

sim_time = 500

normalizers = {
    "fundamental": float(np.mean(historical_prices[:sim_time + 1])),
    "invt": 10,
    "reward": 100.0,
}

env = MMEnv(
    num_background_agents=20,
    sim_time=sim_time,
    lam=0.05,
    lamMM=0.01,
    q_max=10,
    pv_var=1.0,
    est_var=1.0,
    shade=[0.05, 0.5],
    xi=0.5,
    omega=0.5,
    normalizers=normalizers,
    fundamental_type="historical",
    historical_prices=historical_prices,
    policy=True,          # add this
    seed=42,
)

obs, info = env.reset()
print("Initial obs:", obs)

for _ in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print("reward:", reward, "terminated:", terminated)
    if terminated or truncated:
        break

Initial obs: [0.984      0.95119014 1.         0.98651503 0.        ]
reward: 0.0 terminated: False
reward: 0.21631156286233477 terminated: False
reward: 0.028799972534179687 terminated: False
reward: 0.3177511438473823 terminated: True


\\lightning.bu.binghamton.edu\WISE\sescott1\Masters\thesis\potential_codes\pymarketsim\marketsim\wrappers\metrics.py:71: RuntimeWarning: invalid value encountered in scalar divide
  up = deltas[deltas >= 0].sum() / len(midprices)
\\lightning.bu.binghamton.edu\WISE\sescott1\Masters\thesis\potential_codes\pymarketsim\marketsim\wrappers\metrics.py:72: RuntimeWarning: invalid value encountered in scalar divide
  down = -deltas[deltas < 0].sum() / len(midprices)
